# Contextual Multi-Armed Bandit

For the contextual multi-armed bandit (cMAB) when user information is available (context), we implemented a generalisation of Thompson sampling algorithm ([Agrawal and Goyal, 2014](https://arxiv.org/pdf/1209.3352.pdf)) based on NumPyro.

![title](img/cmab.png)

The following notebook contains an example of usage of the class Cmab, which implements the algorithm above.

In [1]:
import numpy as np

from pybandits.cmab import CmabBernoulli
from pybandits.model import BayesianNeuralNetwork, BnnLayerParams, BnnParams, FeaturesConfig, StudentTArray

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
n_samples = 1000
n_features = 5

First, we need to define the input context matrix $X$ of size ($n\_samples, n\_features$) and the mapping of possible actions $a_i \in A$ to their associated model.

In [3]:
# context
X = 2 * np.random.random_sample((n_samples, n_features)) - 1  # random float in the interval (-1, 1)
print("X: context matrix of shape (n_samples, n_features)")
print(X[:10])

X: context matrix of shape (n_samples, n_features)
[[-0.79739514  0.8387916  -0.10568285 -0.26251145 -0.08613857]
 [-0.43431978 -0.99981406 -0.6347479   0.74774698  0.70820854]
 [ 0.54674353 -0.97180435 -0.65177027  0.94301836 -0.81993041]
 [ 0.63047151  0.32101564 -0.71562907  0.70745512  0.67154111]
 [-0.21805359  0.13880362 -0.8038756  -0.77884089  0.39146822]
 [-0.73642568 -0.47471526  0.36921825 -0.45048332  0.55391624]
 [ 0.20136629  0.85791569 -0.67842549  0.53386747  0.24926152]
 [-0.18213439 -0.86131191 -0.23006377 -0.08532498 -0.36276237]
 [-0.30709779  0.03403135  0.8607558  -0.11526628  0.57823812]
 [ 0.71167975  0.17482862  0.11221951 -0.74156405 -0.0546315 ]]


In [4]:
# define action model
bias = StudentTArray.cold_start(mu=1, sigma=2, shape=1)
weight = StudentTArray.cold_start(shape=(n_features, 1))
layer_params = BnnLayerParams(weight=weight, bias=bias)
model_params = BnnParams(bnn_layer_params=[layer_params])
feature_config = FeaturesConfig(n_features=n_features)

update_method = "VI"
update_kwargs = {"num_steps": 100, "batch_size": 128, "optimizer_type": "adam"}

actions = {
    "a1": BayesianNeuralNetwork(
        model_params=model_params,
        feature_config=feature_config,
        update_method=update_method,
        update_kwargs=update_kwargs,
    ),
    "a2": BayesianNeuralNetwork(
        model_params=model_params,
        feature_config=feature_config,
        update_method=update_method,
        update_kwargs=update_kwargs,
    ),
}

We can now init the bandit given the mapping of actions $a_i$ to their model.

In [5]:
# init contextual Multi-Armed Bandit model
cmab = CmabBernoulli(actions=actions)

The predict function below returns the action selected by the bandit at time $t$: $a_t = argmax_k P(r=1|\beta_k, x_t)$. The bandit selects one action per each sample of the contect matrix $X$.

In [6]:
# predict action
pred_actions, _, _ = cmab.predict(X)
print("Recommended action: {}".format(pred_actions[:10]))

Recommended action: ['a1', 'a2', 'a1', 'a2', 'a1', 'a1', 'a2', 'a2', 'a1', 'a1']


Now, we observe the rewards and the context from the environment. In this example rewards and the context are randomly simulated.

In [7]:
# simulate reward from environment
simulated_rewards = np.random.randint(2, size=n_samples).tolist()
print("Simulated rewards: {}".format(simulated_rewards[:10]))

Simulated rewards: [0, 0, 0, 0, 1, 0, 0, 1, 0, 0]


Finally, we update the model providing per each action sample: (i) its context $x_t$ (ii) the action $a_t$ selected by the bandit, (iii) the corresponding reward $r_t$.

In [8]:
# update model
cmab.update(context=X, actions=pred_actions, rewards=simulated_rewards)

SVI:   0%|          | 0/34 [00:00<?, ?it/s]

SVI:   3%|▎         | 1/34 [00:00<00:31,  1.06it/s]

SVI:   3%|▎         | 1/34 [00:00<00:31,  1.06it/s, loss=1927.4371]

SVI:   6%|▌         | 2/34 [00:00<00:30,  1.06it/s, loss=2386.6846]

SVI:   9%|▉         | 3/34 [00:00<00:29,  1.06it/s, loss=2747.1360]

...

SVI:  91%|█████████ | 31/34 [00:01<00:02,  1.03it/s, loss=1631.3627]

SVI:  94%|█████████▍| 32/34 [00:01<00:01,  1.03it/s, loss=1869.8883]

SVI:  97%|█████████▋| 33/34 [00:01<00:00,  1.03it/s, loss=2175.2119]

SVI: 100%|██████████| 34/34 [00:01<00:00, 22.28it/s, loss=2175.2119]

SVI: 100%|██████████| 34/34 [00:01<00:00, 22.28it/s, loss=2267.9556]